# Expressing the same purpose in Python and Julia


In [1]:
from pathlib import Path
import ast
import re
import sys
import tomllib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
import os
import RiskLabAI
PYTHON_PACKAGE = Path(RiskLabAI.__file__).resolve().parent
PYTHON_SOURCE = PYTHON_PACKAGE.parent
JULIA_ROOT_VALUE = os.environ.get("RISKLABAI_JULIA_ROOT", "")
JULIA_PACKAGE = Path(JULIA_ROOT_VALUE).expanduser() if JULIA_ROOT_VALUE else None
TABLES = BOOK_ROOT / "outputs" / "tables" / "A_risklabai_design"
FIGURES = BOOK_ROOT / "outputs" / "figures" / "A_risklabai_design"
BOOK_FIGURES = FIGURES
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "A_risklabai_design"
for directory in [TABLES, FIGURES, BOOK_FIGURES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BOOK_ROOT))
from book_figure_style import PALETTE, apply_book_figure_style, finish_axis

apply_book_figure_style()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return f"{int(value):,}"
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\centering\n\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    else:
        latex = latex.replace("\\begin{tabular}", "\\centering\n\\begin{tabular}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def save_figure(figure, stem):
    for directory in [FIGURES, BOOK_FIGURES]:
        figure.savefig(directory / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["white"], transparent=False)
        figure.savefig(directory / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["white"], transparent=False)
    display(figure)
    plt.close(figure)


def add_box(axis, x, y, width, height, text, facecolor, edgecolor, textcolor=None):
    patch = FancyBboxPatch((x - width / 2, y - height / 2), width, height, boxstyle="round,pad=0.04,rounding_size=0.08", facecolor=facecolor, edgecolor=edgecolor, linewidth=1.3, zorder=3)
    axis.add_patch(patch)
    axis.text(x, y, text, ha="center", va="center", color=textcolor or PALETTE["navy"], fontsize=10.2, zorder=4)


## Public release boundaries


In [2]:
assert JULIA_PACKAGE is not None and (JULIA_PACKAGE / "Project.toml").is_file(), "Set RISKLABAI_JULIA_ROOT to a public RiskLabAI.jl checkout"
julia_project = tomllib.loads((JULIA_PACKAGE / "Project.toml").read_text(encoding="utf-8"))
python_version = RiskLabAI.__version__
julia_version = julia_project["version"]
boundaries = pd.DataFrame([
    {"Library": "RiskLabAI for Python", "Version used": python_version, "Status in this appendix": "Released public baseline", "Executable here": "Yes"},
    {"Library": "RiskLabAI for Julia", "Version used": julia_version, "Status in this appendix": "Confirmed public baseline", "Executable here": "No, source inspection only"},
])
save_table(boundaries, "python_julia_release_boundaries", "Release boundaries used for the software design comparison.", "tab:appendix-release-boundaries", fit=True)


,Library,Version used,Status in this appendix,Executable here
0,RiskLabAI for Python,3.2.0,Released public baseline,Yes
1,RiskLabAI for Julia,1.1.0,Confirmed public baseline,"No, source inspection only"


,Library,Version used,Status in this appendix,Executable here
0,RiskLabAI for Python,3.2.0,Released public baseline,Yes
1,RiskLabAI for Julia,1.1.0,Confirmed public baseline,"No, source inspection only"


## Native language choices


In [3]:
python_root = (PYTHON_PACKAGE / "__init__.py").read_text(encoding="utf-8")
python_registry = (PYTHON_PACKAGE / "core" / "registry.py").read_text(encoding="utf-8")
python_base = (PYTHON_PACKAGE / "core" / "base.py").read_text(encoding="utf-8")
python_strategy = (PYTHON_PACKAGE / "features" / "feature_importance" / "feature_importance_strategy.py").read_text(encoding="utf-8")
python_bars = (PYTHON_PACKAGE / "data" / "structures" / "abstract_bars.py").read_text(encoding="utf-8")
python_deprecation = (PYTHON_PACKAGE / "_deprecation.py").read_text(encoding="utf-8")
julia_root = (JULIA_PACKAGE / "src" / "RiskLabAI.jl").read_text(encoding="utf-8")
julia_validation = (JULIA_PACKAGE / "src" / "Validation" / "Validation.jl").read_text(encoding="utf-8")
julia_cross = (JULIA_PACKAGE / "src" / "Validation" / "CrossValidators.jl").read_text(encoding="utf-8")
julia_types = (JULIA_PACKAGE / "src" / "Data" / "Structures" / "types.jl").read_text(encoding="utf-8")
julia_bars = (JULIA_PACKAGE / "src" / "Data" / "Structures" / "standard_bars.jl").read_text(encoding="utf-8")
julia_equations = (JULIA_PACKAGE / "src" / "Pde" / "Equations.jl").read_text(encoding="utf-8")
julia_pde = (JULIA_PACKAGE / "src" / "Pde" / "Pde.jl").read_text(encoding="utf-8")

comparison = pd.DataFrame([
    {"Purpose": "Present a public entry point", "Python design": "Module front door with deferred loading", "Julia design": "Main module with explicit exports"},
    {"Purpose": "Choose behavior", "Python design": "Common interfaces and replaceable strategy objects", "Julia design": "Types selected through multiple dispatch"},
    {"Purpose": "Construct a named component", "Python design": "Registry maps a name to a constructor", "Julia design": "Direct construction and dispatched functions"},
    {"Purpose": "Vary a bar stopping rule", "Python design": "Shared base class with a supplied rule", "Julia design": "Metric types with dispatched conditions"},
    {"Purpose": "Provide neural solvers", "Python design": "Deferred import and dependency guard", "Julia design": "Direct dependencies and exported solver types"},
    {"Purpose": "Preserve older public use", "Python design": "Deprecation wrappers", "Julia design": "Legacy exports retained at the public module"},
])
save_table(comparison, "python_julia_design_comparison", "How the public Python and Julia libraries express similar software purposes.", "tab:appendix-python-julia-design", fit=True)


,Purpose,Python design,Julia design
0,Present a public entry point,Module front door with deferred loading,Main module with explicit exports
1,Choose behavior,Common interfaces and replaceable strategy obj...,Types selected through multiple dispatch
2,Construct a named component,Registry maps a name to a constructor,Direct construction and dispatched functions
3,Vary a bar stopping rule,Shared base class with a supplied rule,Metric types with dispatched conditions
4,Provide neural solvers,Deferred import and dependency guard,Direct dependencies and exported solver types
5,Preserve older public use,Deprecation wrappers,Legacy exports retained at the public module


,Purpose,Python design,Julia design
0,Present a public entry point,Module front door with deferred loading,Main module with explicit exports
1,Choose behavior,Common interfaces and replaceable strategy obj...,Types selected through multiple dispatch
2,Construct a named component,Registry maps a name to a constructor,Direct construction and dispatched functions
3,Vary a bar stopping rule,Shared base class with a supplied rule,Metric types with dispatched conditions
4,Provide neural solvers,Deferred import and dependency guard,Direct dependencies and exported solver types
5,Preserve older public use,Deprecation wrappers,Legacy exports retained at the public module


## Comparing structure without claiming identical implementations


In [4]:
mapping = comparison[["Python design", "Purpose", "Julia design"]].copy()
python_labels = ["Module front door with\ndeferred loading", "Common interfaces and\nreplaceable strategies", "Registry maps a name\nto a constructor", "Shared base class with\na supplied rule", "Deferred import and\ndependency guard", "Deprecation wrappers"]
julia_labels = ["Main module with\nexplicit exports", "Types selected through\nmultiple dispatch", "Direct construction and\ndispatched functions", "Metric types with\ndispatched conditions", "Direct dependencies and\nexported solver types", "Legacy exports retained\nat the public module"]
mapping.to_csv(FIGURE_DATA / "python_julia_design_map.csv", index=False)
figure, axis = plt.subplots(figsize=(11.2, 7.4), constrained_layout=True)
y_values = np.arange(len(mapping))[::-1]
for row_index, (y, row) in enumerate(zip(y_values, mapping.itertuples(index=False))):
    axis.add_patch(FancyArrowPatch((2.1, y), (3.25, y), arrowstyle="-|>", mutation_scale=13, color=PALETTE["light_gray"], linewidth=1.2))
    axis.add_patch(FancyArrowPatch((6.75, y), (7.9, y), arrowstyle="-|>", mutation_scale=13, color=PALETTE["light_gray"], linewidth=1.2))
    add_box(axis, 0.7, y, 2.8, 0.68, python_labels[row_index], "#1E3A5F", PALETTE["blue"])
    add_box(axis, 5.0, y, 3.2, 0.68, row[1], PALETTE["near_white"], PALETTE["gray"])
    add_box(axis, 9.3, y, 2.8, 0.68, julia_labels[row_index], "#134E4A", PALETTE["teal"])
axis.text(0.7, len(mapping) + 0.05, "Python", ha="center", color=PALETTE["blue"], fontsize=12, fontweight="bold")
axis.text(5.0, len(mapping) + 0.05, "Shared purpose", ha="center", color=PALETTE["navy"], fontsize=12, fontweight="bold")
axis.text(9.3, len(mapping) + 0.05, "Julia", ha="center", color=PALETTE["teal"], fontsize=12, fontweight="bold")
axis.set(xlim=(-1.0, 11.0), ylim=(-0.8, len(mapping) + 0.65), title="Different Language Designs for the Same Public Purpose")
axis.axis("off")
save_figure(figure, "python_julia_design_map")


<Figure size 1120x740 with 1 Axes>

## Tests


In [5]:
assert python_version == "3.2.0"
assert julia_version == "1.1.0"
assert "def __getattr__" in python_root
assert "class Registry" in python_registry
assert "class Estimator(Protocol)" in python_base
assert "class FeatureImportanceStrategy" in python_strategy
assert "class AbstractBars" in python_bars
assert "def deprecated" in python_deprecation
assert "module RiskLabAI" in julia_root
assert "multiple dispatch" in julia_validation
assert "struct KFoldCV" in julia_cross
assert "abstract type Metric" in julia_types
assert "bar_construction_condition" in julia_bars
assert "abstract type Equation" in julia_equations
assert 'include("DeepBSDESolver.jl")' in julia_pde
assert all(dependency in julia_project["deps"] for dependency in ["Lux", "Optimisers", "Zygote"])
